# Vector Spaces, Rank & Linear Systems

向量空间、秩与线性方程组。从线性组合到四个基本子空间，从秩-零度定理到方程组可解性，全程配代码与可视化。

## 0. 环境配置与导入

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
print("PyTorch version:", torch.__version__)
torch.manual_seed(42)

## 1. 线性组合与张成空间

### 1.1 线性组合

给定向量组 $v_1, v_2, \ldots, v_k \in \mathbb{R}^n$ 和标量 $c_1, c_2, \ldots, c_k$，它们的**线性组合**为：

$$
c_1 v_1 + c_2 v_2 + \cdots + c_k v_k
$$

### 1.2 张成空间

向量组 $\{v_1, \ldots, v_k\}$ 的**张成空间**（span）是所有线性组合构成的集合：

$$
\text{span}(v_1, \ldots, v_k) = \left\{ \sum_{i=1}^k c_i v_i \mid c_i \in \mathbb{R} \right\}
$$

张成空间是一个向量空间（子空间）。

In [ ]:
# 2D 中的线性组合与张成
v1 = torch.tensor([1.0, 2.0])
v2 = torch.tensor([3.0, 1.0])

print("v1 =", v1.tolist())
print("v2 =", v2.tolist())

# 不同系数的线性组合
print("\n一些线性组合:")
for c1, c2 in [(1, 0), (0, 1), (1, 1), (2, -1), (-1, 3)]:
    combo = c1 * v1 + c2 * v2
    print(f"  {c1}*v1 + {c2}*v2 = {combo.tolist()}")

# 可视化：v1, v2 张成整个 2D 平面
fig, ax = plt.subplots(figsize=(6, 6))
# 画很多随机线性组合，应该填满整个平面
for _ in range(200):
    c1 = torch.randn(1).item()
    c2 = torch.randn(1).item()
    combo = c1 * v1 + c2 * v2
    ax.scatter(combo[0], combo[1], c='blue', alpha=0.2, s=10)
ax.arrow(0, 0, v1[0], v1[1], head_width=0.2, color='red', linewidth=2, label='v1')
ax.arrow(0, 0, v2[0], v2[1], head_width=0.2, color='green', linewidth=2, label='v2')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title("span(v1, v2) = 整个 2D 平面（v1, v2 不共线）")
ax.set_xlim(-8, 8); ax.set_ylim(-8, 8)
plt.tight_layout()
plt.show()

In [ ]:
# 共线向量只能张成一条直线
v1 = torch.tensor([1.0, 2.0])
v2 = 2 * v1  # v2 与 v1 共线

print("v1 =", v1.tolist())
print("v2 = 2*v1 =", v2.tolist())
print("v1 和 v2 共线 → span 是一条直线")

fig, ax = plt.subplots(figsize=(6, 6))
for _ in range(200):
    c1 = torch.randn(1).item()
    c2 = torch.randn(1).item()
    combo = c1 * v1 + c2 * v2
    ax.scatter(combo[0], combo[1], c='blue', alpha=0.2, s=10)
ax.arrow(0, 0, v1[0], v1[1], head_width=0.2, color='red', linewidth=2, label='v1')
ax.arrow(0, 0, v2[0], v2[1], head_width=0.2, color='green', linewidth=2, label='v2=2v1')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title("span(v1, v2) = 一条直线（v1, v2 共线）")
ax.set_xlim(-8, 8); ax.set_ylim(-8, 8)
plt.tight_layout()
plt.show()

## 2. 线性无关与线性相关

### 2.1 定义

向量组 $\{v_1, \ldots, v_k\}$ 是**线性无关**（linearly independent）的，当且仅当：

$$
c_1 v_1 + c_2 v_2 + \cdots + c_k v_k = 0 \implies c_1 = c_2 = \cdots = c_k = 0
$$

即只有全零系数才能得到零向量。

否则称为**线性相关**（linearly dependent）：存在不全为零的系数使得线性组合为零向量。

**等价条件**：
- 矩阵 $A = [v_1 \ v_2 \ \cdots \ v_k]$ 的秩 = k（列满秩）⟺ 线性无关
- $Ax = 0$ 只有零解 ⟺ 线性无关
- $\det(A) \neq 0$（方阵时）⟺ 线性无关

In [ ]:
# 判断线性无关：矩阵的秩
# 例1：线性无关
A_indep = torch.tensor([[1.0, 2.0],
                         [3.0, 4.0]])
rank1 = torch.linalg.matrix_rank(A_indep)
print("A_indep =\n", A_indep)
print("rank(A_indep) =", rank1.item(), "(=列数2 → 线性无关)")
print("det(A_indep) =", torch.det(A_indep).item(), "(≠0 → 线性无关)")

# 例2：线性相关（第二列 = 2*第一列）
A_dep = torch.tensor([[1.0, 2.0],
                       [2.0, 4.0]])
rank2 = torch.linalg.matrix_rank(A_dep)
print("\nA_dep =\n", A_dep)
print("rank(A_dep) =", rank2.item(), "(<列数2 → 线性相关)")
print("det(A_dep) =", torch.det(A_dep).item(), "(=0 → 线性相关)")

# 找到非零系数使得 c1*v1 + c2*v2 = 0
# v2 = 2*v1 → 2*v1 - 1*v2 = 0
print("\n非零系数: c1=2, c2=-1 → 2*v1 - v2 =", (2*A_dep[:,0] - A_dep[:,1]).tolist())

In [ ]:
# 3D 中的线性无关判断
# 三个向量，如果第三个在前两个的张成空间内，则线性相关
v1 = torch.tensor([1.0, 0.0, 0.0])
v2 = torch.tensor([0.0, 1.0, 0.0])
v3_dep = v1 + v2  # 在 v1, v2 的张成空间内
v3_indep = torch.tensor([0.0, 0.0, 1.0])  # 不在张成空间内

A_dep3 = torch.stack([v1, v2, v3_dep], dim=1)
A_indep3 = torch.stack([v1, v2, v3_indep], dim=1)

print("线性相关组（v3 = v1+v2）:")
print("  rank =", torch.linalg.matrix_rank(A_dep3).item(), "(<3 → 相关)")
print("  det =", torch.det(A_dep3).item())

print("\n线性无关组（标准基）:")
print("  rank =", torch.linalg.matrix_rank(A_indep3).item(), "(=3 → 无关)")
print("  det =", torch.det(A_indep3).item())

# 用零空间判断：Ax=0 有非零解 → 线性相关
null_dep = torch.linalg.svd(A_dep3)[2][2:]  # SVD 的 V 的后几行
print("\n相关组的零空间向量:", null_dep.tolist())
print("A @ null =", (A_dep3 @ null_dep.T).tolist(), "(≈0 → 非零解)")

## 3. 基与维度

### 3.1 基

向量空间 $V$ 的**基**（basis）是 $V$ 中一个线性无关且张成 $V$ 的向量组。

**性质**：
- 基中的向量个数是唯一的，称为空间的**维度**（dimension），记为 $\dim(V)$
- $\mathbb{R}^n$ 的标准基是 $e_1, e_2, \ldots, e_n$，维度为 n
- 空间中任意向量都可以唯一表示为基的线性组合

### 3.2 坐标

给定基 $B = \{b_1, \ldots, b_n\}$，向量 $v$ 在基 $B$ 下的**坐标**是唯一的系数 $(c_1, \ldots, c_n)$，使得 $v = c_1 b_1 + \cdots + c_n b_n$。

In [ ]:
# 不同的基，同一个向量的不同坐标
v = torch.tensor([3.0, 4.0])

# 标准基
e1 = torch.tensor([1.0, 0.0])
e2 = torch.tensor([0.0, 1.0])
print("向量 v =", v.tolist())
print("标准基坐标: (3, 4) → 3*e1 + 4*e2 =", (3*e1 + 4*e2).tolist())

# 另一个基
b1 = torch.tensor([1.0, 1.0])
b2 = torch.tensor([1.0, -1.0])
B = torch.stack([b1, b2], dim=1)
# 求坐标：B @ c = v → c = B^{-1} @ v
c = torch.linalg.solve(B, v)
print("\n基 b1 =", b1.tolist(), ", b2 =", b2.tolist())
print("坐标 c =", c.tolist())
print(f"验证: {c[0]:.1f}*b1 + {c[1]:.1f}*b2 =", (c[0]*b1 + c[1]*b2).tolist())

# 可视化
fig, ax = plt.subplots(figsize=(6, 6))
ax.arrow(0, 0, v[0], v[1], head_width=0.2, color='black', linewidth=3, label='v')
ax.arrow(0, 0, e1[0], e1[1], head_width=0.15, color='blue', linewidth=1.5, label='e1')
ax.arrow(0, 0, e2[0], e2[1], head_width=0.15, color='blue', linewidth=1.5, label='e2')
ax.arrow(0, 0, b1[0], b1[1], head_width=0.15, color='red', linewidth=1.5, label='b1')
ax.arrow(0, 0, b2[0], b2[1], head_width=0.15, color='red', linewidth=1.5, label='b2')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title("同一个向量在不同基下的坐标")
ax.set_xlim(-1, 5); ax.set_ylim(-2, 5)
plt.tight_layout()
plt.show()

## 4. 四个基本子空间

对于任意矩阵 $A \in \mathbb{R}^{m \times n}$，存在四个基本子空间：

| 子空间 | 符号 | 定义 | 所在空间 | 维度 |
|--------|------|------|---------|------|
| 列空间 | $C(A)$ | A 的列的所有线性组合 | $\mathbb{R}^m$ | $r$ |
| 行空间 | $C(A^T)$ | A 的行的所有线性组合 | $\mathbb{R}^n$ | $r$ |
| 零空间 | $N(A)$ | 满足 $Ax=0$ 的所有 x | $\mathbb{R}^n$ | $n-r$ |
| 左零空间 | $N(A^T)$ | 满足 $A^T y=0$ 的所有 y | $\mathbb{R}^m$ | $m-r$ |

其中 $r = \text{rank}(A)$。

**正交关系**：
- 行空间 $C(A^T)$ ⊥ 零空间 $N(A)$（在 $\mathbb{R}^n$ 中互为正交补）
- 列空间 $C(A)$ ⊥ 左零空间 $N(A^T)$（在 $\mathbb{R}^m$ 中互为正交补）

In [ ]:
# 四个基本子空间的维度
A = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])  # 2x3 矩阵

m, n = A.shape
r = torch.linalg.matrix_rank(A).item()

print(f"A 是 {m}x{n} 矩阵，秩 r = {r}")
print(f"\n四个基本子空间:")
print(f"  列空间 C(A):     R^{m}, 维度 = r = {r}")
print(f"  行空间 C(A^T):   R^{n}, 维度 = r = {r}")
print(f"  零空间 N(A):     R^{n}, 维度 = n-r = {n-r}")
print(f"  左零空间 N(A^T): R^{m}, 维度 = m-r = {m-r}")

# 用 SVD 求四个子空间的基
U, S, Vh = torch.linalg.svd(A)
print(f"\nSVD 奇异值: {S.tolist()}（非零个数 = r = {r}）")
print(f"\n列空间的基（U 的前 r 列）:\n{U[:, :r]}")
print(f"\n行空间的基（Vh 的前 r 行）:\n{Vh[:r, :]}")
print(f"\n零空间的基（Vh 的后 n-r 行）:\n{Vh[r:, :]}")
print(f"\n左零空间的基（U 的后 m-r 列）:\n{U[:, r:]}")

In [ ]:
# 验证正交关系：行空间 ⊥ 零空间
A = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
U, S, Vh = torch.linalg.svd(A)
r = 2

# 行空间的基
row_basis = Vh[:r, :]  # [2, 3]
# 零空间的基
null_basis = Vh[r:, :]  # [1, 3]

print("行空间的基:\n", row_basis)
print("\n零空间的基:\n", null_basis)

# 验证正交：行空间的每个基向量与零空间的每个基向量点积为 0
print("\n正交性验证（行空间基 · 零空间基）:")
for i in range(r):
    for j in range(null_basis.shape[0]):
        dot = torch.dot(row_basis[i], null_basis[j])
        print(f"  row[{i}] · null[{j}] = {dot.item():.6f} (≈0)")

# 验证零空间：A @ x = 0
print("\n验证零空间: A @ null_basis.T =")
print(A @ null_basis.T, "(≈0)")

# 验证列空间 ⊥ 左零空间
col_basis = U[:, :r]
left_null_basis = U[:, r:]
print("\n列空间基 · 左零空间基:")
for i in range(r):
    for j in range(left_null_basis.shape[1]):
        dot = torch.dot(col_basis[:, i], left_null_basis[:, j])
        print(f"  col[{i}] · left_null[{j}] = {dot.item():.6f} (≈0)")

## 5. 秩与秩-零度定理

### 5.1 秩

矩阵 $A$ 的**秩**（rank）是：
- A 的线性无关列的最大个数（列秩）
- A 的线性无关行的最大个数（行秩）
- 列空间（或行空间）的维度
- 非零奇异值的个数

**重要事实**：列秩 = 行秩 = rank(A)。

### 5.2 秩-零度定理

> **秩-零度定理（Rank-Nullity Theorem）**：
> 对于 $A \in \mathbb{R}^{m \times n}$，
> $$
> \text{rank}(A) + \text{nullity}(A) = n
> $$
> 其中 $\text{nullity}(A) = \dim N(A)$ 是零空间的维度（零度）。

即：列空间维度 + 零空间维度 = 列数。

In [ ]:
# 秩-零度定理验证
matrices = [
    torch.randn(5, 7),           # 满秩 5
    torch.tensor([[1., 2., 3.], [2., 4., 6.], [1., 1., 1.]]),  # 秩 2
    torch.zeros(3, 4),            # 秩 0
    torch.eye(5),                  # 秩 5
    torch.randn(4, 4)[:, :2] @ torch.randn(2, 4),  # 秩 2（低秩构造）
]

print(f"{'矩阵形状':>10} {'秩':>4} {'零度':>4} {'秩+零度':>6} {'=列数?':>6}")
print("-" * 40)
for A in matrices:
    m, n = A.shape
    r = torch.linalg.matrix_rank(A).item()
    nullity = n - r
    print(f"{str(A.shape):>10} {r:>4} {nullity:>4} {r+nullity:>6} {'✓' if r+nullity==n else '✗':>6}")

# 用 SVD 验证：非零奇异值个数 = 秩
print("\n--- 用 SVD 验证秩 = 非零奇异值个数 ---")
A = torch.randn(5, 5)[:, :3] @ torch.randn(3, 5)  # 构造秩 3
S = torch.linalg.svdvals(A)
print("奇异值:", S.tolist())
print("非零奇异值个数（>1e-10）:", (S > 1e-10).sum().item())
print("matrix_rank:", torch.linalg.matrix_rank(A).item())

In [ ]:
# 秩的几何意义：列空间的维度
# 秩 1 矩阵：所有列都在一条直线上
A_rank1 = torch.tensor([[1.0, 2.0, 3.0],
                         [2.0, 4.0, 6.0]])  # 第二行 = 2*第一行，列也共线

print("秩 1 矩阵 A =\n", A_rank1)
print("rank(A) =", torch.linalg.matrix_rank(A_rank1).item())
print("列空间是 R^2 中的一条直线")

# 可视化列空间
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 秩1：列都在一条直线上
ax = axes[0]
for j in range(A_rank1.shape[1]):
    col = A_rank1[:, j]
    ax.arrow(0, 0, col[0], col[1], head_width=0.3, linewidth=2, label=f'col{j+1}')
# 列空间直线
t = torch.linspace(-1, 4, 100)
ax.plot(t, 2*t, 'r--', alpha=0.5, label='列空间（直线）')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title("秩 1：列空间是一条直线（维度=1）")
ax.set_xlim(-1, 4); ax.set_ylim(-1, 7)

# 秩2：列张成整个平面
A_rank2 = torch.tensor([[1.0, 0.0, 1.0],
                         [0.0, 1.0, 1.0]])  # 秩 2
ax = axes[1]
for j in range(A_rank2.shape[1]):
    col = A_rank2[:, j]
    ax.arrow(0, 0, col[0], col[1], head_width=0.15, linewidth=2, label=f'col{j+1}')
# 随机线性组合填满平面
for _ in range(100):
    c = torch.randn(3)
    combo = A_rank2 @ c
    ax.scatter(combo[0], combo[1], c='blue', alpha=0.1, s=5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title("秩 2：列空间是整个平面（维度=2）")
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)

plt.tight_layout()
plt.show()

## 6. 线性方程组可解性

对于线性方程组 $Ax = b$，其中 $A \in \mathbb{R}^{m \times n}$：

### 6.1 可解性条件

$Ax = b$ 有解 $\iff$ $b \in C(A)$（b 在 A 的列空间中）。

等价条件：$\text{rank}(A) = \text{rank}([A \mid b])$（增广矩阵的秩不变）。

### 6.2 解的个数

| 条件 | 解的个数 |
|------|---------|
| $b \notin C(A)$ | 无解 |
| $b \in C(A)$ 且 $\text{rank}(A) = n$（列满秩） | 唯一解 |
| $b \in C(A)$ 且 $\text{rank}(A) < n$ | 无穷多解（解空间 = 特解 + 零空间） |

### 6.3 解的结构

当有无穷多解时，通解 = 一个特解 + 齐次方程 $Ax=0$ 的通解（零空间中的任意向量）。

In [ ]:
# 三种情况：无解、唯一解、无穷多解

# 情况1：无解（b 不在列空间中）
A1 = torch.tensor([[1.0, 1.0], [1.0, 1.0]])  # 秩 1，列空间是直线 x=y
b1 = torch.tensor([1.0, 2.0])  # 不在直线 x=y 上
rank_A1 = torch.linalg.matrix_rank(A1).item()
rank_aug1 = torch.linalg.matrix_rank(torch.cat([A1, b1.unsqueeze(1)], dim=1)).item()
print("=== 情况1：无解 ===")
print(f"A rank = {rank_A1}, 增广矩阵 rank = {rank_aug1}")
print(f"b 在列空间中? {rank_A1 == rank_aug1}")
try:
    x = torch.linalg.solve(A1, b1)
    print("解:", x.tolist())
except RuntimeError as e:
    print("solve 报错:", str(e)[:80])

# 情况2：唯一解（列满秩，b 在列空间中）
A2 = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # 秩 2 = n
b2 = torch.tensor([5.0, 11.0])
rank_A2 = torch.linalg.matrix_rank(A2).item()
print("\n=== 情况2：唯一解 ===")
print(f"A rank = {rank_A2} = n=2（列满秩）")
x2 = torch.linalg.solve(A2, b2)
print("唯一解 x =", x2.tolist())
print("验证 A@x =", (A2 @ x2).tolist(), "== b?", torch.allclose(A2 @ x2, b2))

# 情况3：无穷多解（秩 < n，b 在列空间中）
A3 = torch.tensor([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0]])  # 秩 1 < n=3
b3 = torch.tensor([6.0, 12.0])  # b = 2*第一列，在列空间中
rank_A3 = torch.linalg.matrix_rank(A3).item()
rank_aug3 = torch.linalg.matrix_rank(torch.cat([A3, b3.unsqueeze(1)], dim=1)).item()
print("\n=== 情况3：无穷多解 ===")
print(f"A rank = {rank_A3} < n=3, 增广 rank = {rank_aug3}")
print(f"b 在列空间中? {rank_A3 == rank_aug3}")
print(f"零空间维度 = n-r = {3-rank_A3} → 解有 {3-rank_A3} 个自由度")

# 用 lstsq 求一个特解
x3, _, _, _ = torch.linalg.lstsq(A3, b3)
print("一个特解 x_p =", x3.tolist())
print("验证 A@x_p =", (A3 @ x3).tolist())

# 零空间向量
U, S, Vh = torch.linalg.svd(A3)
null_basis = Vh[rank_A3:, :]
print("\n零空间的基:\n", null_basis)
# 特解 + 任意零空间向量也是解
for t_val in [1.0, -2.0, 3.5]:
    x_general = x3 + t_val * null_basis[0]
    print(f"  x_p + {t_val}*null = {x_general.tolist()}, A@x = {(A3 @ x_general).tolist()}")

In [ ]:
# 可视化：2D 中两条直线的交点情况
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

x = torch.linspace(-3, 3, 100)

# 唯一解：两条不平行直线相交于一点
ax = axes[0]
ax.plot(x, 2 - x, 'b-', label='x+y=2')
ax.plot(x, 0.5*x + 0.5, 'r-', label='x-2y=-1')
ax.scatter([1], [1], c='green', s=100, zorder=5, label='唯一解 (1,1)')
ax.set_title("唯一解：rank(A)=2=n")
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend()
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)

# 无解：两条平行直线不相交
ax = axes[1]
ax.plot(x, 1 - x, 'b-', label='x+y=1')
ax.plot(x, 2 - x, 'r-', label='x+y=2')
ax.set_title("无解：rank(A)=1 < rank(增广)=2")
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend()
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)

# 无穷多解：两条直线重合
ax = axes[2]
ax.plot(x, 2 - x, 'b-', linewidth=3, label='x+y=2')
ax.plot(x, 2 - x, 'r--', linewidth=2, label='2x+2y=4（同一条线）')
ax.set_title("无穷多解：rank(A)=rank(增广)=1 < n=2")
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend()
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)

plt.suptitle("线性方程组 Ax=b 的三种解的情况", fontsize=14)
plt.tight_layout()
plt.show()

## 7. 应用：秩与数据矩阵的低秩结构

在机器学习中，数据矩阵 $X \in \mathbb{R}^{n \times d}$（n 个样本，d 个特征）的秩反映了数据的内在维度。

- **满秩数据**：特征之间没有线性依赖，需要全部 d 个维度描述
- **低秩数据**：特征之间存在线性依赖，数据实际上位于更低维的子空间中
- **低秩近似**：用截断 SVD 找到最佳秩-k 近似，去除噪声和冗余

这是 PCA、推荐系统、矩阵补全、图像压缩等的核心思想。

In [ ]:
# 低秩数据的例子：数据实际上位于低维子空间
torch.manual_seed(42)
n_samples = 300
true_dim = 2  # 真实内在维度

# 生成 2D 数据，然后嵌入到 5D 空间（加随机投影 + 噪声）
Z = torch.randn(n_samples, true_dim)  # 真实 2D 数据
W = torch.randn(true_dim, 5)           # 随机投影矩阵 2→5
X = Z @ W + 0.05 * torch.randn(n_samples, 5)  # 5D 观测数据 + 小噪声

print(f"数据矩阵 X: {X.shape}")
print(f"秩: {torch.linalg.matrix_rank(X).item()}")
print(f"（虽然是 5D 数据，但内在维度是 {true_dim}，秩接近 {true_dim}）")

# 奇异值衰减：前 true_dim 个大，后面小
S = torch.linalg.svdvals(X)
print("\n奇异值:", S.tolist())
print("奇异值平方占比（能量）:", (S**2 / (S**2).sum() * 100).tolist())

# 可视化奇异值衰减
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(range(1, 6), S.numpy(), color='steelblue')
axes[0].set_xlabel("奇异值序号")
axes[0].set_ylabel("奇异值")
axes[0].set_title("奇异值衰减（前2个大，后3个小=噪声）")
axes[0].grid(True, alpha=0.3)

# 累积能量
cum_energy = torch.cumsum(S**2, dim=0) / (S**2).sum() * 100
axes[1].plot(range(1, 6), cum_energy.numpy(), 'bo-', linewidth=2)
axes[1].axhline(y=95, color='r', linestyle='--', label='95% 能量')
axes[1].set_xlabel("保留的奇异值个数 k")
axes[1].set_ylabel("累积能量 (%)")
axes[1].set_title("累积能量：k=2 时已保留 >95% 能量")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(80, 101)

plt.tight_layout()
plt.show()

# 用 k=2 做低秩近似（PCA 降维）
U, S, Vh = torch.linalg.svd(X, full_matrices=False)
k = 2
X_approx = U[:, :k] @ torch.diag(S[:k]) @ Vh[:k, :]
error = torch.norm(X - X_approx) / torch.norm(X) * 100
print(f"\n秩-{k} 近似相对误差: {error:.2f}%")
print(f"压缩率: 原始 {X.numel()} 参数 → 近似 {k*(X.shape[0]+X.shape[1])} 参数")

In [ ]:
# 秩-1 矩阵的结构：外积形式
# 任何秩-1 矩阵都可以写成 u @ v.T（外积）
u = torch.tensor([1.0, 2.0, 3.0])
v = torch.tensor([4.0, 5.0])
A_rank1 = u.unsqueeze(1) @ v.unsqueeze(0)  # 外积

print("u =", u.tolist())
print("v =", v.tolist())
print("\nA = u @ v.T =\n", A_rank1)
print("rank(A) =", torch.linalg.matrix_rank(A_rank1).item(), "(秩 1)")

# 验证：每一行都是 v 的倍数，倍数是 u 的元素
print("\n验证行结构:")
for i in range(3):
    print(f"  行{i} = {A_rank1[i].tolist()} = {u[i].item()} * v = {(u[i]*v).tolist()}")

# 秩-r 矩阵 = r 个秩-1 矩阵之和
print("\n--- 秩-r 矩阵 = r 个外积之和 ---")
A = torch.randn(4, 4)
U, S, Vh = torch.linalg.svd(A)
r = torch.linalg.matrix_rank(A).item()
print(f"随机 4x4 矩阵的秩 = {r}")

# 逐步累加外积
A_reconstructed = torch.zeros(4, 4)
for i in range(r):
    outer = S[i] * U[:, i:i+1] @ Vh[i:i+1, :]
    A_reconstructed += outer
    error = torch.norm(A - A_reconstructed).item()
    print(f"  累加前 {i+1} 个外积后，误差 = {error:.6f}")

print(f"\n全部 {r} 个外积之和 = A? {torch.allclose(A_reconstructed, A, atol=1e-5)}")

## 课后练习

### 基础题

1. 判断下列向量组是否线性无关，并说明理由：
   - $\{(1,2), (2,4)\}$
   - $\{(1,0,0), (0,1,0), (0,0,1), (1,1,1)\}$
   - $\{(1,2,3), (4,5,6), (7,8,9)\}$
   用 `torch.linalg.matrix_rank` 验证。

2. 对矩阵 $A = \begin{pmatrix} 1 & 2 & 3 \\ 2 & 4 & 6 \\ 1 & 1 & 1 \end{pmatrix}$，求四个基本子空间的维度和基（用 SVD），验证行空间⊥零空间、列空间⊥左零空间。

3. 验证秩-零度定理：创建 5 个不同形状和秩的矩阵，对每个验证 $\text{rank}(A) + \text{nullity}(A) = n$。

### 线性方程组

4. 给出三个线性方程组的例子，分别对应无解、唯一解、无穷多解。对每个例子：
   - 计算 rank(A) 和 rank(增广矩阵)
   - 说明解的个数
   - 无穷多解时，写出通解形式（特解 + 零空间）

5. 对于超定方程组 $Ax=b$（m>n，列满秩），证明最小二乘解 $x = (A^T A)^{-1} A^T b$ 满足正规方程 $A^T A x = A^T b$。用代码验证一个 5×2 的例子。

### 应用题

6. 生成一个 100×10 的数据矩阵，其中真实内在维度为 3（用 3 个隐因子生成，加噪声）。用 SVD 分析：
   - 奇异值衰减图
   - 确定保留多少个主成分能保留 95% 能量
   - 用秩-k 近似重构数据，计算相对误差
   - 可视化降维到 2D 的结果

7. 证明：如果 A 是 m×n 矩阵且 rank(A)=r，则 A 可以分解为 r 个秩-1 矩阵之和。用一个 4×3 秩-2 矩阵验证。

### 综合题

8. 四个基本子空间的维度关系总结表：对 m×n 秩 r 的矩阵，填写四个子空间的所在空间和维度，并解释为什么行空间和零空间的维度之和为 n。

9. 用几何直觉解释：为什么当 b 不在 A 的列空间中时，方程组 Ax=b 无解？用 2D 中两条平行直线的例子可视化说明。